Model Lead


In [2]:
# Import library
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt

# Wrapper kompatibilitas: NumPy 1.x pakai np.trapz, NumPy 2.x pakai np.trapezoid
_trapz = getattr(np, 'trapezoid', None) or np.trapz

# Parameter model (dapat diubah)
T1 = 10.0   # akhir fase 1 (akselerasi)
T2 = 60.0   # akhir fase 2 (konstan)
T3 = 80.0   # akhir fase 3 (deselerasi) = T total
V_MAX = 20.0  # kecepatan jelajah (m/s)

print("Parameter model:")
print(f"  Fase 1 (akselerasi)  : 0  <= t <= {T1:.0f}  s,  v(t) = 2t")
print(f"  Fase 2 (konstan)     : {T1:.0f} <  t <= {T2:.0f}  s,  v(t) = {V_MAX:.0f}")
print(f"  Fase 3 (deselerasi)  : {T2:.0f} <  t <= {T3:.0f}  s,  v(t) = 80 - t")


Parameter model:
  Fase 1 (akselerasi)  : 0  <= t <= 10  s,  v(t) = 2t
  Fase 2 (konstan)     : 10 <  t <= 60  s,  v(t) = 20
  Fase 3 (deselerasi)  : 60 <  t <= 80  s,  v(t) = 80 - t


In [3]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt

# implementasi v(t) numerik
def v_num(t_array):
    t_array = np.asarray(t_array, dtype=float)
    return np.piecewise(
        t_array,
        [
            t_array <= 10,
            (t_array > 10) & (t_array <= 60),
            t_array > 60
        ],
        [
            lambda x: 2 * x,
            lambda x: 20.0,
            lambda x: 80 - x
        ]
    )

# sanity check di titik-titik penting
print("Cek nilai v(t) di titik kunci:")
for t_test in [0, 5, 10, 35, 60, 70, 80]:
    print(f"  v({t_test:>2}) = {v_num(t_test):.1f} m/s")

Cek nilai v(t) di titik kunci:
  v( 0) = 0.0 m/s
  v( 5) = 10.0 m/s
  v(10) = 20.0 m/s
  v(35) = 20.0 m/s
  v(60) = 20.0 m/s
  v(70) = 10.0 m/s
  v(80) = 0.0 m/s


In [4]:
# kompatibilitas NumPy 1.x vs 2.x
_trapz = getattr(np, 'trapezoid', None) or np.trapz

# fungsi integral trapesium untuk satu fase
def trapz_phase(t_start, t_end, n):
    tt = np.linspace(t_start, t_end, n + 1)
    vv = v_num(tt)
    return _trapz(vv, tt)

# hitung untuk dua resolusi
for n in [1_000, 100_000]:
    s1_n = trapz_phase(0,  10, n)
    s2_n = trapz_phase(10, 60, n)
    s3_n = trapz_phase(60, 80, n)
    s_tot = s1_n + s2_n + s3_n

    print(f"n = {n:,} titik per fase:")
    print(f"  s1 (akselerasi)  = {s1_n:.8f} m")
    print(f"  s2 (konstan)     = {s2_n:.8f} m")
    print(f"  s3 (deselerasi)  = {s3_n:.8f} m")
    print(f"  total            = {s_tot:.8f} m")
    print()

n = 1,000 titik per fase:
  s1 (akselerasi)  = 100.00000000 m
  s2 (konstan)     = 1000.00000000 m
  s3 (deselerasi)  = 200.00000000 m
  total            = 1300.00000000 m

n = 100,000 titik per fase:
  s1 (akselerasi)  = 100.00000000 m
  s2 (konstan)     = 1000.00000000 m
  s3 (deselerasi)  = 200.00000000 m
  total            = 1300.00000000 m



In [5]:
# validasi hasil (selisih absolut dan relatif, serta alasan perbedaan).

# hitung nilai analitik dengan SymPy 
t = sp.Symbol('t', real=True)

S1_ANALYTIC = float(sp.integrate(2*t,        (t, 0,  10)))  # = 100.0 m
S2_ANALYTIC = float(sp.integrate(20,         (t, 10, 60)))  # = 1000.0 m
S3_ANALYTIC = float(sp.integrate(80 - t,     (t, 60, 80)))  # = 200.0 m
S_TOTAL_ANALYTIC = S1_ANALYTIC + S2_ANALYTIC + S3_ANALYTIC  # = 1300.0 m

print("Hasil analitik (SymPy):")
print(f"  Fase 1 : {S1_ANALYTIC} m")
print(f"  Fase 2 : {S2_ANALYTIC} m")
print(f"  Fase 3 : {S3_ANALYTIC} m")
print(f"  Total  : {S_TOTAL_ANALYTIC} m")

# error analisis untuk berbagai nilai n 
n_values = [1_000, 100_000]

print("\n Error Analysis")
for n in n_values:
    s1_n    = trapz_phase(0,  10, n)
    s2_n    = trapz_phase(10, 60, n)
    s3_n    = trapz_phase(60, 80, n)
    s_tot_n = s1_n + s2_n + s3_n

    hasil = [
        ("Fase 1 (akselerasi)",  s1_n,    S1_ANALYTIC),
        ("Fase 2 (konstan)",     s2_n,    S2_ANALYTIC),
        ("Fase 3 (deselerasi)",  s3_n,    S3_ANALYTIC),
        ("Total",               s_tot_n, S_TOTAL_ANALYTIC),
    ]

    print(f"\n n = {n:,} titik")
    for label, s_n, s_a in hasil:
        err_abs = abs(s_n - s_a) # hitung error absolut
        err_rel = err_abs / abs(s_a) if s_a != 0 else 0.0 # hitung error relatif

        print(f"  {label}")
        print(f"    Numerik  : {s_n:.6f} m")
        print(f"    Analitik : {s_a:.6f} m")
        print(f"    Err abs  : {err_abs:.2e} m")
        print(f"    Err rel  : {err_rel:.2e} ")




Hasil analitik (SymPy):
  Fase 1 : 100.0 m
  Fase 2 : 1000.0 m
  Fase 3 : 200.0 m
  Total  : 1300.0 m

 Error Analysis

 n = 1,000 titik
  Fase 1 (akselerasi)
    Numerik  : 100.000000 m
    Analitik : 100.000000 m
    Err abs  : 0.00e+00 m
    Err rel  : 0.00e+00 
  Fase 2 (konstan)
    Numerik  : 1000.000000 m
    Analitik : 1000.000000 m
    Err abs  : 0.00e+00 m
    Err rel  : 0.00e+00 
  Fase 3 (deselerasi)
    Numerik  : 200.000000 m
    Analitik : 200.000000 m
    Err abs  : 0.00e+00 m
    Err rel  : 0.00e+00 
  Total
    Numerik  : 1300.000000 m
    Analitik : 1300.000000 m
    Err abs  : 0.00e+00 m
    Err rel  : 0.00e+00 

 n = 100,000 titik
  Fase 1 (akselerasi)
    Numerik  : 100.000000 m
    Analitik : 100.000000 m
    Err abs  : 0.00e+00 m
    Err rel  : 0.00e+00 
  Fase 2 (konstan)
    Numerik  : 1000.000000 m
    Analitik : 1000.000000 m
    Err abs  : 0.00e+00 m
    Err rel  : 0.00e+00 
  Fase 3 (deselerasi)
    Numerik  : 200.000000 m
    Analitik : 200.000000 m
    E

Penjelasan hasil validasi : 
Hasil validasi menunjukkan bahwa nilai integral numerik yang diperoleh menggunakan metode trapesium sangat dekat dengan hasil analitik yang dihitung menggunakan SymPy. Error absolut dan error relatif pada setiap fase bernilai nol atau mendekati nol. Karena fungsi yang digunakan hanya fungsi linear dan konstan. Metode trapesium memberikan hasil eksak untuk fungsi linear, sehingga hasil numerik sama dengan hasil analitik.